# Minecraft Decision Lab — sequential QLoRA training

Train Qwen3-1.7B, download its results, then train Qwen2.5-7B-Instruct on the same single Colab GPU. Each model has a locked 300-case base evaluation and adapter evaluation.

**Saved status: GPU execution has not been performed in this notebook.** Empty output cells are intentional. Results are created only by executing the stages below. This is a synthetic text-state decision prototype, not a Minecraft gameplay benchmark.

Set a caller-supplied deadline through `MC_HARD_DEADLINE_UTC` or the prompt below before starting. The script enforces that deadline and a maximum of seven elapsed GPU-runtime hours. Confirm authorization for a new run separately. Select a GPU runtime before starting. No Drive mount, new account, paid API, or publication step is required.

## 1. Fetch the public project

This installs nothing on the research Mac. All files and model caches remain in the temporary `/content` runtime. Existing runs are reused without replacing their files.

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime, timezone

if not Path("/content").exists():
    raise RuntimeError("Run this notebook in Google Colab, not on the local Mac.")
PROJECT_ROOT = Path("/content/mc-decision-lab")
REPOSITORY = "https://github.com/myconstance1999-pixel/mc-decision-lab.git"
HARD_DEADLINE_UTC = os.environ.get("MC_HARD_DEADLINE_UTC", "").strip() or input("Authorized run deadline (ISO 8601 with UTC offset): ").strip()
if not HARD_DEADLINE_UTC:
    raise ValueError("A caller-supplied deadline is required before starting a run.")
authorized_deadline = datetime.fromisoformat(HARD_DEADLINE_UTC.replace("Z", "+00:00"))
if authorized_deadline.tzinfo is None or authorized_deadline <= datetime.now(timezone.utc):
    raise ValueError("Supply a future deadline with an explicit UTC offset.")
MAX_GPU_HOURS = 7.0
MAX_LENGTH = 512  # The preflight checks BOTH tokenizers before any weights are loaded.

if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(PROJECT_ROOT)], check=True)
if not (PROJECT_ROOT / "training/run.py").exists():
    raise RuntimeError("The public checkout does not yet contain the training entry. Finish publication first.")
os.chdir(PROJECT_ROOT)
os.environ["HF_HOME"] = str(PROJECT_ROOT / ".model-cache")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
budget_path = PROJECT_ROOT / "runs/runtime_budget.json"
budget_path.parent.mkdir(parents=True, exist_ok=True)
if not budget_path.exists():
    budget_path.write_text(json.dumps({
        "started_at": datetime.now(timezone.utc).isoformat(),
        "hard_deadline_utc": HARD_DEADLINE_UTC,
        "max_runtime_hours": MAX_GPU_HOURS,
        "note": "Conservative elapsed-runtime cap including setup. Monitor compute units in Colab."
    }, indent=2) + "\n")
print("Project:", PROJECT_ROOT)
print("Project commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("Python:", sys.version)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=True)

## 2. Install the pinned training stack

The direct dependency versions obey the inspected LlamaFactory commit. Colab's existing matching CUDA PyTorch packages are preserved. Runner stages use separate Python processes, so they do not depend on already-imported notebook packages. Installation and import success do not establish GPU training success.

In [ ]:
LLAMAFACTORY_COMMIT = "47d407d2aabbd52e11d4f34c7a06b9de3e2f0a77"
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "-r", str(PROJECT_ROOT / "training/requirements-colab.txt"),
    f"llamafactory @ git+https://github.com/hiyouga/LlamaFactory.git@{LLAMAFACTORY_COMMIT}",
], check=True)
subprocess.run([
    sys.executable, "-c",
    "import torch, transformers, peft, trl, bitsandbytes, llamafactory; "
    "print('CUDA:', torch.cuda.is_available(), 'PyTorch:', torch.__version__); "
    "from training.run import verify_llamafactory; verify_llamafactory(); print('Pinned imports passed')"
], check=True, cwd=PROJECT_ROOT)

## 3. Define the visible stage runner

A failed command stops its cell. Existing completed predictions are preserved, and incomplete predictions resume from their matching test prefix. Training resumes only a checkpoint with optimizer, scheduler, and RNG state.

In [ ]:
def run_stage(model, stage):
    command = [sys.executable, "-u", "training/run.py", "--model", model, "--stage", stage,
               "--max-length", str(MAX_LENGTH), "--epochs", "2",
               "--max-gpu-hours", str(MAX_GPU_HOURS), "--hard-deadline-utc", HARD_DEADLINE_UTC]
    print(" ".join(command), flush=True)
    subprocess.run(command, check=True, cwd=PROJECT_ROOT)

for required in ("train", "val", "test"):
    path = PROJECT_ROOT / "data" / f"{required}.json"
    if not path.exists():
        raise RuntimeError(f"Dataset is not ready: {path}")
print({split: len(json.loads((PROJECT_ROOT / "data" / f"{split}.json").read_text()))
       for split in ("train", "val", "test")})

## 4. Check both tokenizers and all answer masks

This stage fetches tokenizer files only. Every full prompt and answer must fit the selected limit for both models; the script rejects silent truncation. It checks actual LlamaFactory labels and the masked Qwen3 nonthinking prefix. If this fails, correct the data or choose 1,024 tokens before any training, then rerun. Once the preflight passes, keep its data and cutoff fixed.

In [ ]:
run_stage("qwen3-1.7b", "preflight")

## 5. Qwen3-1.7B: baseline → training → adapter

Execute these cells in order. The model uses `template: qwen3` with `enable_thinking: false`. Only validation loss selects a checkpoint; the test set is not used for selection.

In [ ]:
run_stage("qwen3-1.7b", "baseline")

In [ ]:
run_stage("qwen3-1.7b", "train")

In [ ]:
run_stage("qwen3-1.7b", "adapter")

## 6. Download the first completed model

The ZIP contains its adapter, raw predictions, metrics, logs, and provenance. It excludes base weights, caches, and optimizer checkpoints. Complete the normal browser download before moving on.

In [ ]:
run_stage("qwen3-1.7b", "package")
from google.colab import files
files.download(str(PROJECT_ROOT / "artifacts/qwen3-1.7b.zip"))

## 7. Qwen2.5-7B-Instruct: baseline → training → adapter

The first model is released from memory after each stage. The second model uses `template: qwen` and the same data, quantization, effective batch, epoch limit, and decoding protocol. Do not start this section if the remaining authorized time or compute units are insufficient.

In [ ]:
run_stage("qwen2.5-7b", "baseline")

In [ ]:
run_stage("qwen2.5-7b", "train")

In [ ]:
run_stage("qwen2.5-7b", "adapter")

## 8. Download the second completed model

In [ ]:
run_stage("qwen2.5-7b", "package")
files.download(str(PROJECT_ROOT / "artifacts/qwen2.5-7b.zip"))

## 9. Inspect measured metrics

These values are read from actual evaluator output. Missing files remain missing evidence; no placeholder score is supplied. Decision accuracy, skill accuracy, joint correctness, and invalid output rate measure this synthetic text task, not actual game execution.

In [ ]:
for model in ("qwen3-1.7b", "qwen2.5-7b"):
    for kind in ("base", "adapter"):
        path = PROJECT_ROOT / "results" / f"{model}_{kind}_metrics.json"
        print(f"\n{model} / {kind}")
        print(path.read_text() if path.exists() else "NOT RUN: metrics are unavailable")

## Optional: preserve a resumable checkpoint before ending the runtime

Publication ZIPs contain adapter weights only. For a later exact training resume, preserve the most recent checkpoint plus its training identity and pinned data hashes. This optional ZIP can be larger. Restore it into the same project-relative paths; use the same dataset and dependency versions. No cloud storage mount is required.

In [ ]:
# Run this cell only if training was interrupted and an exact resume is needed.
import zipfile
from training.run import latest_checkpoint
resume_zip = PROJECT_ROOT / "artifacts/resume-checkpoints.zip"
resume_zip.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(resume_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for model in ("qwen3-1.7b", "qwen2.5-7b"):
        run_dir = PROJECT_ROOT / "runs" / model
        checkpoint = latest_checkpoint(run_dir / "adapter")
        if checkpoint:
            for path in checkpoint.rglob("*"):
                if path.is_file():
                    archive.write(path, path.relative_to(PROJECT_ROOT))
            for path in run_dir.glob("*.json"):
                archive.write(path, path.relative_to(PROJECT_ROOT))
    for path in (PROJECT_ROOT / "runs/preflight.json", PROJECT_ROOT / "runs/runtime_budget.json"):
        if path.exists():
            archive.write(path, path.relative_to(PROJECT_ROOT))
files.download(str(resume_zip))

## Finish

Confirm that both artifact downloads finished and inspect their contents. Release the Colab runtime when no further authorized GPU work remains. Publishing to GitHub Releases and updating the homepage are separate delivery steps; this notebook does neither.

Source: [pinned LlamaFactory](https://github.com/hiyouga/LlamaFactory/tree/47d407d2aabbd52e11d4f34c7a06b9de3e2f0a77). The notebook structure and cell syntax are locally checked; GPU execution and native Colab presentation must be verified in the live run.